# Notebook 08b -- Textbook Oracle Algorithms

**Prerequisites:** Notebooks 01-07. Familiarity with superposition, measurement, and the oracle model.

The earliest quantum algorithms were designed not to solve practical problems,
but to prove that quantum computers can outperform classical ones in
well-defined settings. These **textbook algorithms** operate in the **oracle
model**: they treat a function $f$ as a black box and ask how many queries
are needed to learn some property of $f$.

By the end of this notebook you will be able to:

1. **Explain** how Deutsch-Jozsa distinguishes constant from balanced oracles in one query.
2. **Implement** Bernstein-Vazirani to recover a hidden bitstring.
3. **Describe** Simon's algorithm and how it finds hidden periods with $O(n)$ queries.
4. **Compare** the quantum query complexity of each algorithm to its classical counterpart.

All three algorithms share the same **H-Oracle-H sandwich** pattern:
superposition, oracle query, interference, measure. The key difference
is what property of $f$ each algorithm extracts.

In [1]:
import (
	"context"
	"fmt"

	"github.com/janpfeifer/gonb/gonbui"
	"github.com/splch/goqu/algorithm/textbook"
	"github.com/splch/goqu/circuit/builder"
	"github.com/splch/goqu/circuit/draw"
	"github.com/splch/goqu/sim/statevector"
	"github.com/splch/goqu/viz"
)

## The Deutsch-Jozsa Algorithm

**Problem:** Given a black-box function $f: \{0,1\}^n \to \{0,1\}$ that is
promised to be either **constant** (same output for all inputs) or **balanced**
(outputs 0 for exactly half the inputs and 1 for the other half), determine
which type it is.

**Classical lower bound:** Any deterministic classical algorithm needs
$2^{n-1} + 1$ queries in the worst case -- it must check just over half the
inputs before it can be certain.

**Quantum solution:** The Deutsch-Jozsa algorithm answers with **a single
query** by exploiting interference:

1. Prepare all input qubits in superposition with H gates.
2. Apply the oracle once.
3. Apply H gates again and measure.

If $f$ is constant, all input qubits measure $|0\rangle$ (constructive
interference on the all-zeros outcome). If $f$ is balanced, at least one
qubit measures $|1\rangle$ (destructive interference eliminates the all-zeros
outcome).

### Building Deutsch-Jozsa from scratch

The circuit has three stages:
1. **Superposition**: Apply H to all input qubits and X+H to the ancilla
2. **Oracle**: Apply the black-box function
3. **Interference**: Apply H to all input qubits and measure

This H-Oracle-H sandwich is the core pattern shared by Deutsch-Jozsa,
Bernstein-Vazirani, and Simon's algorithm.

In [2]:
%%
// Build Deutsch-Jozsa from scratch for a balanced oracle (mask=101)
b := builder.New("dj-manual", 4).WithClbits(3) // 3 input + 1 ancilla, 3 classical bits

// Step 1: Prepare ancilla in |-> and inputs in |+>
b.X(3).H(3)  // ancilla in |->
b.H(0).H(1).H(2)  // inputs in superposition

// Step 2: Oracle — balanced with mask 101 (CNOT from inputs 0,2 to ancilla)
b.CNOT(0, 3).CNOT(2, 3)

// Step 3: Interference — H on input qubits and measure
b.H(0).H(1).H(2)
b.Measure(0, 0).Measure(1, 1).Measure(2, 2)

c, _ := b.Build()
fmt.Println("Manual Deutsch-Jozsa circuit:")
gonbui.DisplayHTML(draw.SVG(c))

// Run and check
sim := statevector.New(4)
counts, _ := sim.Run(c, 1000)
fmt.Println("\nMeasurement counts:", counts)

// For balanced oracle, the all-zeros outcome |000> should be absent
_, hasAllZeros := counts["000"]
fmt.Printf("All-zeros outcome present: %v (expected: false for balanced)\n", hasAllZeros)

Manual Deutsch-Jozsa circuit:

Measurement counts: map[0101:517 1101:483]
All-zeros outcome present: false (expected: false for balanced)


q0 
 
 q1 
 
 q2 
 
 q3 
 
 X 
 H 
 H 
 H 
 H 
 
 
 
 
 
 
 H 
 H 
 H 
 M 
 M 
 M

Now let's see the same algorithm via Goqu's built-in `textbook` package, which handles all this construction for you:

In [3]:
%%
ctx := context.Background()

// Run Deutsch-Jozsa with a constant oracle (f(x) = 0 for all x)
result, err := textbook.DeutschJozsa(ctx, textbook.DJConfig{
	NumQubits: 3,
	Oracle:    textbook.ConstantOracle(0),
	Shots:     1000,
})
if err != nil {
	panic(err)
}

fmt.Println("=== Constant Oracle (f(x) = 0) ===")
fmt.Println("Circuit:")
gonbui.DisplayHTML(draw.SVG(result.Circuit))
fmt.Println("Measurement counts:", result.Counts)
fmt.Printf("Is constant: %v\n", result.IsConstant)
fmt.Println("\nAll qubits measure 0 -- constructive interference on |000>.")

=== Constant Oracle (f(x) = 0) ===
Circuit:
Measurement counts: map[000:1000]
Is constant: true

All qubits measure 0 -- constructive interference on |000>.


q0 
 
 q1 
 
 q2 
 
 q3 
 
 X 
 H 
 H 
 H 
 H 
 H 
 H 
 H 
 M 
 M 
 M

In [4]:
%%
ctx := context.Background()

// Run Deutsch-Jozsa with a balanced oracle (mask = 0b101)
// This oracle computes f(x) = popcount(x AND 101) mod 2
result, err := textbook.DeutschJozsa(ctx, textbook.DJConfig{
	NumQubits: 3,
	Oracle:    textbook.BalancedOracle(0b101),
	Shots:     1000,
})
if err != nil {
	panic(err)
}

fmt.Println("=== Balanced Oracle (mask = 101) ===")
fmt.Println("Measurement counts:", result.Counts)
fmt.Printf("Is constant: %v\n", result.IsConstant)
fmt.Println("\nThe all-zeros outcome is absent -- the algorithm detects 'balanced'.")

=== Balanced Oracle (mask = 101) ===
Measurement counts: map[101:1000]
Is constant: false

The all-zeros outcome is absent -- the algorithm detects 'balanced'.


In [5]:
%%
ctx := context.Background()

// Histogram comparing constant vs balanced results
constResult, _ := textbook.DeutschJozsa(ctx, textbook.DJConfig{
	NumQubits: 3,
	Oracle:    textbook.ConstantOracle(0),
	Shots:     1000,
})
balResult, _ := textbook.DeutschJozsa(ctx, textbook.DJConfig{
	NumQubits: 3,
	Oracle:    textbook.BalancedOracle(0b101),
	Shots:     1000,
})

svgConst := viz.Histogram(constResult.Counts, viz.WithTitle("Deutsch-Jozsa: Constant Oracle"))
svgBal := viz.Histogram(balResult.Counts, viz.WithTitle("Deutsch-Jozsa: Balanced Oracle"))
gonbui.DisplayHTML(svgConst)
gonbui.DisplayHTML(svgBal)

Deutsch-Jozsa: Constant Oracle 
 
 
 
 0 
 
 200 
 
 400 
 
 600 
 
 800 
 
 1000 
 
 000

## The Bernstein-Vazirani Algorithm

**Problem:** Given an oracle for $f(x) = s \cdot x \mod 2$ (the dot product
of a hidden string $s$ with the input $x$, modulo 2), find $s$.

**Classical lower bound:** A classical algorithm needs $n$ queries -- one for
each bit of $s$ (query with $x = e_i$ to learn $s_i$).

**Quantum solution:** Bernstein-Vazirani recovers the entire $n$-bit secret
in **a single query**. The circuit is identical to Deutsch-Jozsa (Hadamard,
oracle, Hadamard, measure), but the measurement outcome directly encodes
the secret string $s$.

The key insight: the Hadamard transform converts the phase kickback from
the oracle into a bitstring in the computational basis.

In [6]:
%%
ctx := context.Background()

// Run Bernstein-Vazirani to find the secret string s = 1011 (decimal 11)
result, err := textbook.BernsteinVazirani(ctx, textbook.BVConfig{
	NumQubits: 4,
	Secret:    0b1011, // secret = "1011"
	Shots:     1000,
})
if err != nil {
	panic(err)
}

fmt.Println("=== Bernstein-Vazirani ===")
fmt.Println("Circuit:")
gonbui.DisplayHTML(draw.SVG(result.Circuit))
fmt.Printf("Secret found: %04b (decimal %d)\n", result.Secret, result.Secret)
fmt.Println("Measurement counts:", result.Counts)
fmt.Println("\nThe secret string 1011 is recovered in a single query!")

=== Bernstein-Vazirani ===
Circuit:
Secret found: 1011 (decimal 11)
Measurement counts: map[1011:1000]

The secret string 1011 is recovered in a single query!


q0 
 
 q1 
 
 q2 
 
 q3 
 
 q4 
 
 X 
 H 
 H 
 H 
 H 
 H 
 
 
 
 
 
 
 
 
 
 H 
 H 
 H 
 H 
 M 
 M 
 M 
 M

In [7]:
%%
ctx := context.Background()

result, _ := textbook.BernsteinVazirani(ctx, textbook.BVConfig{
	NumQubits: 4,
	Secret:    0b1011,
	Shots:     1000,
})

svg := viz.Histogram(result.Counts, viz.WithTitle("Bernstein-Vazirani: Secret = 1011"))
gonbui.DisplayHTML(svg)

Bernstein-Vazirani: Secret = 1011 
 
 
 
 0 
 
 200 
 
 400 
 
 600 
 
 800 
 
 1000 
 
 1011

## Simon's Algorithm

**Problem:** Given an oracle for $f: \{0,1\}^n \to \{0,1\}^n$ with the
promise that there exists a secret period $s$ such that $f(x) = f(y)$ if and
only if $x = y$ or $x = y \oplus s$, find $s$.

**Classical lower bound:** Any classical algorithm requires $\Omega(2^{n/2})$
queries (birthday paradox bound).

**Quantum solution:** Simon's algorithm needs only $O(n)$ quantum queries.
Each query produces a random value $y$ satisfying $y \cdot s = 0 \mod 2$.
After collecting $n-1$ linearly independent equations over GF(2), Gaussian
elimination recovers $s$.

Simon's algorithm is historically significant as the inspiration for Shor's
algorithm -- both use the same pattern of applying a function in superposition
and then performing a Fourier-like transform to extract hidden structure.

In [8]:
%%
ctx := context.Background()

// Run Simon's algorithm with secret period s = 110 (decimal 6)
result, err := textbook.Simon(ctx, textbook.SimonConfig{
	NumQubits: 3,
	Oracle:    textbook.TwoToOneOracle(0b110, 3),
	Shots:     1000,
})
if err != nil {
	panic(err)
}

fmt.Println("=== Simon's Algorithm ===")
fmt.Println("Circuit (last round):")
gonbui.DisplayHTML(draw.SVG(result.Circuit))
fmt.Printf("Recovered period: %03b (decimal %d)\n", result.Period, result.Period)
fmt.Printf("Equations collected: %d\n", len(result.Equations))
for i, eq := range result.Equations {
	fmt.Printf("  y_%d = %03b  (y . s = 0 mod 2)\n", i, eq)
}

=== Simon's Algorithm ===
Circuit (last round):
Recovered period: 110 (decimal 6)
Equations collected: 3
  y_0 = 001  (y . s = 0 mod 2)
  y_1 = 111  (y . s = 0 mod 2)
  y_2 = 110  (y . s = 0 mod 2)


q0 
 
 q1 
 
 q2 
 
 q3 
 
 q4 
 
 q5 
 
 H 
 H 
 H 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 H 
 H 
 H 
 M 
 M 
 M

## Predict, Then Verify

**Question:** Bernstein-Vazirani recovers a hidden bitstring $s$ in one
query by measuring the output of the H-Oracle-H circuit. If we use a
6-qubit oracle with secret $s = 101010$ (decimal 42), what measurement
outcome should dominate?

**Prediction:** The measurement outcome should be exactly $101010$ with
probability 1, because the algorithm deterministically maps the phase
kickback to the secret string. All 1000 shots should yield the same result.

In [9]:
%%
ctx := context.Background()

// Verify: BV with 6 qubits, secret = 101010 (42)
result, err := textbook.BernsteinVazirani(ctx, textbook.BVConfig{
	NumQubits: 6,
	Secret:    0b101010,
	Shots:     1000,
})
if err != nil {
	panic(err)
}

fmt.Printf("Secret found: %06b (decimal %d)\n", result.Secret, result.Secret)
fmt.Printf("Match: %v\n", result.Secret == 42)
fmt.Println("Counts:", result.Counts)
fmt.Println("\nPrediction confirmed: all shots yield 101010.")

Secret found: 101010 (decimal 42)
Match: true
Counts: map[101010:1000]

Prediction confirmed: all shots yield 101010.


## Self-Check Questions

Before attempting the exercises, make sure you can answer these:

1. Why does Deutsch-Jozsa need only 1 query while a classical deterministic algorithm needs 2^(n-1)+1?
2. In Bernstein-Vazirani, why does the measurement outcome directly reveal the secret string rather than some encoded version of it?
3. How does Simon's algorithm achieve an exponential speedup even though each quantum query only produces one linear equation?

---

## Exercises

### Exercise 1 -- Bernstein-Vazirani with a Different Secret

Run the Bernstein-Vazirani algorithm with 5 qubits and secret $s = 10110$
(decimal 22). Verify that the algorithm recovers the secret in a single query.

In [10]:
%%
// Exercise 1: BV with 5 qubits, secret = 10110 (22)
//
// TODO: Run the Bernstein-Vazirani algorithm with:
//   - NumQubits: 5
//   - Secret: 0b10110 (decimal 22)
//   - Shots: 1000
// Then verify that result.Secret == 22.
//
// Hint: Use textbook.BernsteinVazirani(ctx, textbook.BVConfig{...})

// ctx := context.Background()
//
// result, err := textbook.BernsteinVazirani(ctx, textbook.BVConfig{
// 	NumQubits: 5,
// 	Secret:    0b10110, // TODO: set the secret
// 	Shots:     1000,
// })
// if err != nil {
// 	panic(err)
// }
//
// fmt.Printf("Secret found: %05b (decimal %d)\n", result.Secret, result.Secret)
// fmt.Printf("Correct: %v\n", result.Secret == 22)
// fmt.Println("Counts:", result.Counts)
//
// svg := viz.Histogram(result.Counts, viz.WithTitle("Exercise 1: BV Secret = 10110"))
// gonbui.DisplayHTML(svg)

fmt.Println("TODO: Uncomment the code above and fill in the missing parts!")

TODO: Uncomment the code above and fill in the missing parts!


### Exercise 2 -- Deutsch-Jozsa with More Qubits

Run Deutsch-Jozsa with 5 input qubits using a balanced oracle with mask
$11010$ (decimal 26). Verify that the all-zeros outcome is absent.

In [11]:
%%
// Exercise 2: DJ with 5 qubits, balanced oracle mask = 11010 (26)
//
// TODO: Run the Deutsch-Jozsa algorithm with:
//   - NumQubits: 5
//   - Oracle: textbook.BalancedOracle(0b11010)
//   - Shots: 1000
// Then verify that result.IsConstant == false.
//
// Hint: Use textbook.DeutschJozsa(ctx, textbook.DJConfig{...})

// ctx := context.Background()
//
// result, err := textbook.DeutschJozsa(ctx, textbook.DJConfig{
// 	NumQubits: 5,
// 	Oracle:    textbook.BalancedOracle(0b11010),
// 	Shots:     1000,
// })
// if err != nil {
// 	panic(err)
// }
//
// fmt.Printf("Is constant: %v (expected: false)\n", result.IsConstant)
// fmt.Println("Counts:", result.Counts)
//
// svg := viz.Histogram(result.Counts, viz.WithTitle("Exercise 2: DJ Balanced (mask=11010)"))
// gonbui.DisplayHTML(svg)

fmt.Println("TODO: Uncomment the code above and fill in the missing parts!")

TODO: Uncomment the code above and fill in the missing parts!


### Exercise 3 -- Simon's Algorithm with a Different Period

Run Simon's algorithm with 4 qubits and secret period $s = 1001$ (decimal 9).
Verify that the algorithm recovers the correct period.

In [12]:
%%
// Exercise 3: Simon's algorithm with 4 qubits, period = 1001 (9)
//
// TODO: Run Simon's algorithm with:
//   - NumQubits: 4
//   - Oracle: textbook.TwoToOneOracle(0b1001, 4)
//   - Shots: 1000
// Then verify that result.Period == 9.
//
// Hint: Use textbook.Simon(ctx, textbook.SimonConfig{...})

// ctx := context.Background()
//
// result, err := textbook.Simon(ctx, textbook.SimonConfig{
// 	NumQubits: 4,
// 	Oracle:    textbook.TwoToOneOracle(0b1001, 4),
// 	Shots:     1000,
// })
// if err != nil {
// 	panic(err)
// }
//
// fmt.Printf("Recovered period: %04b (decimal %d)\n", result.Period, result.Period)
// fmt.Printf("Correct: %v\n", result.Period == 9)
// fmt.Printf("Equations collected: %d\n", len(result.Equations))
// for i, eq := range result.Equations {
// 	fmt.Printf("  y_%d = %04b\n", i, eq)
// }

fmt.Println("TODO: Uncomment the code above and fill in the missing parts!")

TODO: Uncomment the code above and fill in the missing parts!


## Key Takeaways

1. **Oracle model algorithms** demonstrate quantum speedups by treating
   functions as black boxes. The speedup is provable because we can count
   queries exactly.

2. **Deutsch-Jozsa** achieves an exponential query speedup: 1 quantum query
   vs. $2^{n-1}+1$ classical queries to distinguish constant from balanced.

3. **Bernstein-Vazirani** achieves a linear query speedup: 1 quantum query
   vs. $n$ classical queries to recover a hidden bitstring.

4. **Simon's algorithm** achieves an exponential speedup: $O(n)$ quantum
   queries vs. $\Omega(2^{n/2})$ classical queries to find a hidden period.
   It directly inspired Shor's factoring algorithm.

5. All three algorithms share the **H-Oracle-H** pattern: prepare a
   superposition, apply the oracle, interfere, and measure. The quantum
   advantage comes from extracting global properties of $f$ that are
   encoded in the relative phases of the superposition.

---

**Next up:** Notebook 09 -- Grover's Search Algorithm, where we'll harness amplitude amplification for unstructured search.